<h1 style="padding-top: 25px;padding-bottom: 25px;text-align: left; padding-left: 10px; background-color: #DDDDDD;
    color: black;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/Harvard-IACS/2018-CS109A/master/content/styles/iacs.png" height="50px"> <a href='https://harvard-iacs.github.io/2025-AC215/' target='_blank'><strong><font color="#A41034">AC215/E115: MLOps & LLMOps: Production AI Systems</font></strong></a></h1>

# **<font color="#A41034">Tutorial -  Agents - 🧀 The Cheesy Way 🧀</font>**

**Harvard University**<br/>
**Fall 2026**<br/>
**Instructor:** Pavlos Protopapas<br/>


<hr style="height:2pt">

# Multi-agent lab — The Cheese Quiz Guild (ELELEM-Chile 2026, Day 3)

**Time:** 100 minutes · **Stack:** OpenAI Agents SDK · **Pattern:** orchestrator / worker with agents-as-tools

You have seen the Guild run end to end in the Agents 1 lab. Today you rebuild its brain. The specialists are given; the orchestration is yours.

**Learning objectives**
1. Turn an agent into a tool (`Agent.as_tool`) and explain what the orchestrator sees when it calls one.
2. Write orchestrator instructions that enforce an assembly line, a review loop with a cap, and a human-decision point.
3. Inspect the trace (`result.new_items`) to see which specialist was called, in what order, and how many model calls it cost.
4. Decide, with evidence, whether a step should be an agent, a tool, or plain Python.

**Tiers:** Core (parts 1–3) · Stretch (part 4) · Challenge (part 5). Every tier ends by *inspecting* what the SDK did, not just reading the quiz.

> Status: untested draft assembled from the Cheese Quiz tutorial notebook (cells reused verbatim where marked). Run-check before the workshop.


## 0 · Setup (as in the Cheese Quiz notebook)

Add `OPENAI_API_KEY` under Colab → Secrets (🔑) before running.

In [ ]:
# @title Install Packages
# Installing necessary libraries
!pip -q install fastmcp openai-agents > /dev/null 2>&1
#%pip install openai-agents > /dev/null 2>&1

In [ ]:
# @title Imports
import os
import asyncio
import html
import json, random, time, threading
import openai
from openai import OpenAI
from pathlib import Path
from typing import Dict, Any
from dataclasses import dataclass
from typing import List, Optional, Dict, Literal
from jsonschema import validate as js_validate, ValidationError
from fastmcp import FastMCP
from contextlib import AsyncExitStack
from pydantic import BaseModel, Field, ValidationError
from agents import (
    Agent,
    Runner,
    WebSearchTool,
    function_tool,
    SQLiteSession,
    ModelSettings,
    FunctionTool
)
from agents.agent_output import AgentOutputSchemaBase
from agents.mcp import MCPServerStreamableHttp
from agents.exceptions import ModelBehaviorError
from agents.items import ToolCallItem, ToolCallOutputItem

from IPython.display import HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("openai.agents").setLevel(logging.ERROR)

In [ ]:
# @title Set Up OpenAI Key
# Using Google Colab's Secrets feature 🔑
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
import os
os.environ['OPENAI_API_KEY'] = api_key

#Proper client initialization
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

In [ ]:
# @title Download
# Download files
!gdown 1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh # https://drive.google.com/file/d/1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh/view?usp=drive_link
!gdown 1OykEQrwa5cnZgPIVrOdnDeq3txots3H0 # https://drive.google.com/file/d/1OykEQrwa5cnZgPIVrOdnDeq3txots3H0/view?usp=drive_link

In [ ]:
MODEL = "gpt-4o"   # same model as the tutorial; change here only

## 1 · Shared utilities (given)

The quiz JSON schema, the HTML renderer, the Pydantic types and two helper tools. Read `QuizAgentOutputSchema.validate_json` — this is the guardrail that makes the orchestrator's final answer *structurally* safe.

In [ ]:
# @title Some Utils

SCHEMA_PATH = Path("quiz-schema.json")

with open("quiz-ui.html", "r") as f:
    QUIZ_UI = f.read()

class QuizAgentOutputSchema(AgentOutputSchemaBase):
    """
    Custom output schema that:
      - returns your JSON Schema (for Structured Outputs)
      - validates and parses the model's JSON into a Python object
    """

    def __init__(self, schema_path: Path = SCHEMA_PATH):
        self._schema_path = schema_path
        self._schema: Dict[str, Any] = json.loads(
            self._schema_path.read_text(encoding="utf-8")
        )
        # tip: assert top-level shape matches Structured Outputs support
        # (only supported/strict JSON Schema features) per docs:
        # https://platform.openai.com/docs/guides/structured-outputs/supported-schemas
        # The uploaded file uses only supported constructs (object/array/string/integer).

    # --- AgentOutputSchemaBase methods required by the SDK ---
    def is_plain_text(self) -> bool:
        return False

    def name(self) -> str:
        return "QuizJSON"

    def json_schema(self) -> Dict[str, Any]:
        # The SDK will pass this to the model as the response schema.
        return self._schema

    def is_strict_json_schema(self) -> bool:
        # Opt into strict mode so the SDK enforces the subset supported by Structured Outputs.
        return True

    def validate_json(self, json_str: str) -> Any:
        """
        Must return the validated object or raise ModelBehaviorError.
        """
        try:
            obj = json.loads(json_str)
        except Exception as e:
            raise ModelBehaviorError(f"Invalid JSON: {e}") from e

        try:
            js_validate(instance=obj, schema=self._schema)  # runtime guardrail
        except ValidationError as e:
            # Signal to the SDK that the model misbehaved.
            raise ModelBehaviorError(
                f"JSON failed schema validation: {e.message}"
            ) from e

        return obj


def render_quiz_html(
    quiz: Dict[str, Any],
    *,
    title: str | None = None,
    shuffle_questions: bool = False,
    shuffle_options: bool = False,
) -> str:

    topic = title or quiz.get("topic") or "Quiz"

    norm_questions = []
    for idx, q in enumerate(quiz.get("questions", []), start=1):
        # Try both shapes
        q_text = q.get("question") or q.get("text") or ""
        opts = q.get("choices") or q.get("options") or []
        # Keys/letters
        normalized_opts = []
        for opt in opts:
            key = opt.get("key") or opt.get("letter") or ""
            normalized_opts.append({"key": str(key), "text": opt.get("text", "")})
        answer_key = q.get("answer_key") or q.get("answer") or ""
        norm_questions.append(
            {
                "id": q.get("id") or idx,
                "text": q_text,
                "options": normalized_opts,
                "answer": str(answer_key),
            }
        )

    client_payload = {
        "title": topic,
        "questions": norm_questions,
        "settings": {
            "shuffleQuestions": bool(shuffle_questions),
            "shuffleOptions": bool(shuffle_options),
        },
    }

    # Escape + serialize once for safe embedding in a <script> tag
    data_json = json.dumps(client_payload, ensure_ascii=False)

    return QUIZ_UI.replace("{ data_json }", data_json)


# ───────────────────────────
# Shared types
# ───────────────────────────
class CheeseFact(BaseModel):
    statement: str
    source: Optional[str] = None

class QuizChoice(BaseModel):
    key: str
    text: str


class QuizQ(BaseModel):
    question: str
    choices: List[QuizChoice]
    answer_key: str
    source_note: Optional[str] = None


class QuizDraft(BaseModel):
    topic: str
    questions: List[QuizQ]


class ReviewResult(BaseModel):
    approved: bool
    notes: str = ""
    needs_revision_indices: List[int] = Field(default_factory=list)
    needs_user_opinion: bool = False


@dataclass
class GuildContext:
    fact_cache: dict[str, List[CheeseFact]]


@dataclass
class OrchestratorResult:
    response_text: str
    structured_json: dict
    formatted_md: str


# ───────────────────────────
# Helper tools
# ───────────────────────────


@function_tool
def ensure_unique_answers(question: str, choices: List[str]) -> bool:
    norm = [c.strip().lower() for c in choices]
    return len(set(norm)) == len(norm)


@function_tool
def ask_head_cheesemonger(prompt: str, default_decision: bool = True) -> bool:
    # In real UI, ask the user. Here we simulate.
    return default_decision

## **Meet the Agents: The Guild Members**

In this section we build the core agents required for our multi-agent system. Giving our agents distinct personas will make them more specialized in their roles.

#### 🧐 **Professor Stilton, the Fact-Finder**
This agent's primary job is to take the user's topic and use a **search tool** to find interesting, verifiable facts.

#### 🌱 **Cashew, the Plant-Based Pro**
Cashew supplements Professor Stilton's research with up to three additional facts when the topic involves vegan or plant-based cheese.

#### ✍️ **Brie, the Creative Wordsmith**
The Brie Agent takes the raw facts from Professor Stilton and transforms them into engaging quiz questions (multiple choice, true/false, etc.). She's the core "creative" LLM.

#### 🧀 **Cheddar, the Sharp Critic**
Cheddar Agent reviews the questions and answers created by Brie. He checks them against Stilton's original facts for accuracy. He also ensures the questions are clear and not too easy or impossibly hard.

#### 🎨 **Mozzarella, the Master Formatter**
The Mozzarella Agent takes a quiz that is approved by Cheddar and converts it to Markdown format.

#### 🎨 **Ricotta, the Schema Scribe**
Ricotta takes a quiz that is approved by Cheddar and converts it into strict JSON.

### 1a · Given: Professor Stilton (research) — read it, run it

Notice `output_type=List[CheeseFact]`: the specialist returns typed data, not prose.

In [ ]:
# Professor Stilton, the Fact-Finder
stilton_agent = Agent(
    name="Professor Stilton, the Fact-Finder",
    model="gpt-4o",
    handoff_description="Gathers concise, citable cheese facts for a topic.",
    instructions=(
        "You are Professor Stilton. Given a cheese topic, use web search to gather ~5 concise, "
        "verifiable facts suitable for quiz questions. Include short source notes (domain or title). "
    ),
    tools=[WebSearchTool()],
    output_type=List[CheeseFact],
    model_settings=ModelSettings(temperature=0.3, tool_choice="auto"),
)

# Test
result = await Runner.run(stilton_agent, "Cheese topic: brie")
cheese_facts = result.final_output
for fact in cheese_facts:
    print(f"{fact.statement} ({fact.source})")

### 1b · TODO: Brie

In [ ]:
# TODO 1 — Brie, the Creative Wordsmith
# Write the instructions so that Brie returns EXACTLY 5 multiple-choice questions, 4 options (keys a,b,c,d),
# one correct answer, fun but precise, and includes source_note when a fact had a source.
# Output must be a structured QuizDraft (see the types above).
brie_agent = Agent(
    name="Brie, the Creative Wordsmith",
    model=MODEL,
    handoff_description="Writes engaging multiple-choice questions from facts.",
    instructions=(
        "..."   # <-- your instructions
    ),
    output_type=QuizDraft,
    model_settings=ModelSettings(temperature=0.7),
)

# Test on Stilton's facts
_input = "Facts:\n" + "\n".join(f"- {f.statement}" for f in cheese_facts)
result = await Runner.run(brie_agent, input=_input)
quiz_draft = result.final_output
display(quiz_draft)
assert len(quiz_draft.questions) == 5, "Brie must return exactly 5 questions"
assert all(len(q.choices) == 4 for q in quiz_draft.questions), "each question needs 4 choices"

### 1c · Given: Cheddar (critic), Mozzarella (Markdown), Cashew (vegan specialist), Ricotta (strict JSON)

In [ ]:
# Cheddar, the Sharp Critic
cheddar_agent = Agent(
    name="Cheddar, the Sharp Critic",
    handoff_description="Reviews quiz clarity & correctness vs facts; requests fixes.",
    instructions=(
        "Compare each question to the supplied facts. Flag ambiguous, too-easy/too-hard, or ungrounded items. "
        "Use ensure_unique_answers to detect duplicate options. If borderline difficulty or taste-based, "
        "set needs_user_opinion=True. Produce ReviewResult with indices & notes."
    ),
    tools=[ensure_unique_answers, ask_head_cheesemonger],
    output_type=ReviewResult,
    model_settings=ModelSettings(temperature=0.2),
)

# Mozzarella, the Master Formatter
mozzarella_agent = Agent(
    name="Mozzarella, the Master Formatter",
    handoff_description="Formats the approved quiz for delivery.",
    instructions=(
        "Format the final, approved QuizDraft into Markdown:\n"
        "• Title\n• Numbered questions\n• Lettered options (a–d)\n• **Answer Key**\n• Optional sources."
    ),
    output_type=str,
    model_settings=ModelSettings(temperature=0.2),
)

# Cashew Agent, Dynamic specialist
cashew_agent = Agent(
    name="Cashew, the Plant-Based Pro",
    handoff_description="Adds vegan/plant-based cheese expertise to research.",
    instructions=(
        "Augment Stilton with up to 3 extra facts when the topic includes vegan/plant-based."
    ),
    tools=[WebSearchTool()],
    output_type=List[CheeseFact],
    model_settings=ModelSettings(temperature=0.4),
)

# Ricotta Agent
ricotta_agent = Agent(
    name="Ricotta, the Schema Scribe",
    handoff_description="Converts an approved quiz into strict JSON that matches the provided schema.",
    instructions=(
        "You will receive an approved quiz (questions with options and the correct answer). "
        "Output ONLY valid JSON that conforms to the provided schema. No markdown, no comments.\n"
        "RULES:\n"
        "1) Output ONLY raw JSON (no Markdown fences, no commentary).\n"
        "2) Use sequential integer IDs starting at 1.\n"
        "3) Each question MUST have exactly four options with 'letter' keys: 'a', 'b', 'c', 'd'.\n"
        "4) The 'answer' must equal one of the option letters ('a'|'b'|'c'|'d').\n"
        "5) Preserve the original question wording and option texts; do not invent facts.\n"
        "6) Do not include sources or extra fields—only the fields in the schema.\n"
    ),
    output_type=QuizAgentOutputSchema(),  # <- custom schema object
    model_settings=ModelSettings(temperature=0.1),
)

### 1d · TODO: your own specialist

In [ ]:
# TODO 2 — add ONE specialist of your own, e.g. "Gouda, the Historian" (adds 2 facts about the history of the cheese)
# or "Feta, the Difficulty Tuner" (rewrites one question to be harder). Keep the same shape as the agents above:
# a name, a handoff_description, instructions, (optional) tools, and a typed output_type.
my_agent = Agent(
    name="...",
    model=MODEL,
    handoff_description="...",
    instructions="...",
    output_type=List[CheeseFact],   # or another type from section 1
    model_settings=ModelSettings(temperature=0.4),
)
# Quick test on a topic of your choice
result = await Runner.run(my_agent, "Cheese topic: gouda")
print(result.final_output)

## 2 · Agents as tools

An agent becomes a tool the orchestrator can call. The tool *name* and *description* are the only things the orchestrator's model sees when deciding what to call — write them like an API, not like a bio.

In [ ]:
# TODO 3 — wrap every specialist as a tool. One is done for you.
stilton_tool = stilton_agent.as_tool("research_facts", "Find ~5 concise, citable facts for a topic.")
brie_tool      = None   # brie_agent.as_tool("write_quiz", "...")
cheddar_tool   = None   # review the quiz against the facts
mozzarella_tool= None   # format quiz into Markdown
ricotta_tool   = None   # format quiz into JSON (strict schema)
cashew_tool    = None   # vegan specialist
my_tool        = None   # your specialist from TODO 2

tools_list = [t for t in (stilton_tool, cashew_tool, brie_tool, cheddar_tool, ricotta_tool, my_tool) if t is not None]
print([t.name for t in tools_list])
assert len(tools_list) >= 5, "wrap at least Stilton, Brie, Cheddar, Ricotta and Cashew"

## **Agent Orchestrator**

In this section **you** write the orchestrator's instructions. The assembly line it must follow:

1.  **The Order (Input)**: The user requests a quiz on "Italian Cheeses."
2.  **Research (Agent 1)**: The request goes to **Professor Stilton**. He searches for 5 key facts about Pecorino, Parmesan, Mozzarella, etc.
3.  **Creation (Agent 2)**: The facts are passed to **Brie**. She crafts 5 multiple-choice questions based on the facts.
4.  **Verification (Agent 3)**: The draft quiz is sent to **Cheddar**. He flags one question as potentially confusing and sends it back to Brie with a note. Brie revises it and sends it back to Cheddar, who now approves the whole set.
5.  **Formatting (Agent 4)**: The final, approved quiz goes to **Ricotta**, who formats it into structured JSON.
6.  **Delivery (Output)**: The final, polished quiz is presented to the user.


In [ ]:
# TODO 4 — the orchestrator. Its instructions ARE the control flow. They must say:
#   1) research first (and when to also call the vegan specialist / your specialist), merge to ~5 facts
#   2) write the quiz from the facts
#   3) review; if not approved and not a matter of opinion -> ONE revision, then re-review (max 2 rounds)
#      if needs_user_opinion -> call ask_head_cheesemonger and proceed with its decision
#   4) format to JSON last; only the last step returns JSON
orchestrator = Agent(
    name="Cheese Quiz Guild Orchestrator",
    model=MODEL,
    instructions=(
        "Act as the head of the Cheese Quiz Guild. Follow the assembly line strictly:\n"
        "..."
    ),
    tools=tools_list + [ask_head_cheesemonger],
    output_type=QuizAgentOutputSchema(),
    model_settings=ModelSettings(),
)

In [ ]:
async def run_guild(topic: str) -> str:

    # seed the orchestrator with the user's topic
    prompt = f"Customer requests a quiz on: {topic}. Produce the final JSON."
    result = await Runner.run(
        orchestrator, prompt, max_turns=12
    )
    quiz = result.final_output
    html_str = render_quiz_html(quiz, shuffle_questions=False, shuffle_options=False)
    return HTML(html_str)

## 3 · Run it — then look at the trace, not just the quiz

In [ ]:
# Create a quiz
await run_guild("Greek cheeses")

In [ ]:
# Inspect what actually happened. Which tools were called, in what order, how many model turns?
result = await Runner.run(orchestrator, "Customer requests a quiz on: Greek cheeses. Produce the final JSON.", max_turns=12)
calls = [it for it in result.new_items if isinstance(it, ToolCallItem)]
print("model turns:", len(result.raw_responses))
for it in calls:
    print("tool call ->", it.raw_item.name)
# TODO 5 — answer in the cell below:
#   a) How many review rounds happened? b) Was your specialist called? Why / why not?
#   c) Which step could be plain Python instead of an agent? (hint: ensure_unique_answers)

## 4 · Stretch — handoffs instead of agents-as-tools

Rebuild the same flow with `handoffs=[...]`: the conversation *moves* to the specialist and does not come back. Observe what you lose (the orchestrator's control over step order) and what you gain (one less hop per step).

In [ ]:
# TODO 6 — a triage agent that hands off to Brie or Stilton depending on whether facts are already supplied
triage = Agent(
    name="Guild Triage",
    model=MODEL,
    instructions="If the user supplies facts, hand off to Brie. Otherwise hand off to Professor Stilton.",
    handoffs=[stilton_agent, brie_agent],
)
r = await Runner.run(triage, "Cheese topic: manchego")
print(type(r.final_output), r.last_agent.name)
# Compare: who answered? Could the triage agent have called review afterwards? (No — that is the point of handoffs.)

## 5 · Challenge — a cost guard and a regression test

The orchestrator has `max_turns=12`. Lower it until the run fails; then write the smallest guard that keeps the Guild inside budget without breaking the quiz. Finally, write a 3-case regression test (three topics) that asserts: 5 questions, 4 options each, answer in a–d, no duplicate options.

In [ ]:
# TODO 7
topics = ["French cheeses", "Vegan cheeses", "Cheeses from Star Wars"]
for t in topics:
    res = await Runner.run(orchestrator, f"Customer requests a quiz on: {t}. Produce the final JSON.", max_turns=8)
    quiz = res.final_output
    # assert ...
    print(t, "->", len(quiz.get("questions", [])), "questions,", len(res.raw_responses), "model turns")